# Exercise: Prophet as an Auto-Tuned Classical Model

Prophet automates trend and seasonality detection. Fit a Prophet model with temperature as a regressor, inspect its components, and compare its forecast to the ARIMA models from earlier.

Prophet decomposes the signal into trend + yearly seasonality + regressor effects. Its intervals are simulation-based rather than analytic, so they may differ in width from SARIMAX even when point forecasts agree.

Implement the three tasks marked with `# YOUR CODE HERE`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX

HORIZON = 12

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date").asfreq("MS")
train_df = df.loc[:"2023-12-01"]
train_demand = train_df["demand_mwh"]
train_temp = train_df["avg_temp_f"]

seasonal_avg_temp = train_df.groupby(train_df.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_avg_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS"),
)

# ARIMA baseline for comparison
arima_fit = SARIMAX(train_demand, order=(2, 1, 1)).fit(disp=False)
arima_fc = arima_fit.get_forecast(HORIZON)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int()
print("Data and ARIMA baseline ready.")

## Task 1: Fit Prophet with Temperature

Create a Prophet model with `yearly_seasonality=True`, `weekly_seasonality=False`, `daily_seasonality=False`. Add `avg_temp_f` as a regressor. Fit it to the training data. Return the fitted model.

**Hint:** Reset the index, rename columns to `ds` and `y`.

In [ ]:
def fit_prophet(train_df):
    # YOUR CODE HERE
    raise NotImplementedError

prophet_model = fit_prophet(train_df)

## Task 2: Generate Prophet Forecast

Use `make_future_dataframe` and `predict`. Return the last `horizon` rows of the prediction with a DatetimeIndex matching the forecast dates.

**Hint:** After calling `predict`, set the index to match forecast dates.

In [ ]:
def prophet_forecast(model, train_df, future_temp, horizon):
    # YOUR CODE HERE
    raise NotImplementedError

prophet_pred = prophet_forecast(prophet_model, train_df, future_temp, HORIZON)

## Task 3: Plot Prophet Components

Use `prophet_model.plot_components()` to visualize the trend and seasonality decomposition.

**Hint:** You need the full forecast DataFrame (training + future periods) for `plot_components`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_demand.index, train_demand.values, color="black", label="Historical")
ax.plot(arima_mean.index, arima_mean.values, color="tab:blue", linestyle="--", label="ARIMA")
ax.fill_between(arima_mean.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1], color="tab:blue", alpha=0.1)
ax.plot(prophet_pred.index, prophet_pred["yhat"].values, color="tab:green", linestyle="--", label="Prophet")
ax.fill_between(prophet_pred.index, prophet_pred["yhat_lower"], prophet_pred["yhat_upper"], color="tab:green", alpha=0.1)
ax.set_ylabel("Demand (MWh)")
ax.legend()
plt.tight_layout()
plt.show()